In [ ]:
%matplotlib agg

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import random

from src.model import (
    load_data, explore_data, split_data, EDA, data_cleaning,
    encode_data, select_model, compare_ensembles,
    tune_hyperparameters, important_features, evaluate_model
)

In [2]:
seed = random.randint(1000, 9999)
print("Random seed: ", seed)

ROOT = Path.cwd()
DATA_PATH = ROOT / "data" / "term-deposit-marketing-2020.csv"

Random seed:  9655


In [3]:
term_deposit_df = load_data(DATA_PATH)
numeric_df, categorical_df = explore_data(term_deposit_df)

       age           job   marital  education default  balance housing loan  \
0       58    management   married   tertiary      no     2143     yes   no   
1       44    technician    single  secondary      no       29     yes   no   
2       33  entrepreneur   married  secondary      no        2     yes  yes   
3       47   blue-collar   married    unknown      no     1506     yes   no   
4       33       unknown    single    unknown      no        1      no   no   
39995   53    technician   married   tertiary      no      395      no   no   
39996   30    management    single   tertiary      no     3340      no   no   
39997   54         admin  divorced  secondary      no      200      no   no   
39998   34    management   married   tertiary      no     1047      no   no   
39999   38    technician   married  secondary      no     1442     yes   no   

        contact  day month  duration  campaign    y  
0       unknown    5   may       261         1   no  
1       unknown    5  

In [4]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    term_deposit_df, target="y", seed=seed
)

Size of the training set: 24000
Size of the validation set: 12800
Size of the testing set: 3200


In [5]:
df_tv = EDA(X_train, y_train, X_val, y_val, categorical_df, numeric_df)


Categorical Value Counts (Train + Val)

job
blue-collar      8615
management       7517
technician       6309
admin            4119
services         3598
retired          1319
self-employed    1313
entrepreneur     1295
unemployed       1014
housemaid         997
student           487
unknown           217


marital
married     22468
single       9974
divorced     4358


education
secondary    19323
tertiary     10323
primary       5747
unknown       1407


default
no     36055
yes      745


housing
yes    22071
no     14729


loan
no     30439
yes     6361


contact
cellular     22892
unknown      11764
telephone     2144


month
may    12481
jul     5854
aug     4829
jun     4338
nov     3285
apr     2498
feb     2113
jan     1081
mar      236
oct       73
dec       12


y
no     34136
yes     2664




In [6]:
term_deposit_df, cat_mode, num_bounds = data_cleaning(
    df_tv, categorical_df, numeric_df
)

In [7]:
X_train, X_val, y_train, y_val, le_dict, scaler, le_y = encode_data(
    X_train, X_val, y_train, y_val, categorical_df, numeric_df
)

In [8]:
models, predictions = select_model(X_train, X_val, y_train, y_val)
print(models)

  0%|          | 0/31 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 1738, number of negative: 22262
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000725 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 692
[LightGBM] [Info] Number of data points in the train set: 24000, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.072417 -> initscore=-2.550146
[LightGBM] [Info] Start training from score -2.550146

Best model for minority class: NearestCentroid
                               Accuracy  Balanced Accuracy  ROC AUC  F1 Score  \
Model                                                                           
NearestCentroid                    0.88               0.80     0.80      0.90   
XGBClassifier                      0.94               0.70     0.70      0.93   
Perceptron                         0.89               0.70     0.70     

In [ ]:
fitted_models, results_df = compare_ensembles(
    X_train, y_train, X_val, y_val, seed, cv=5
)
print(results_df)

In [ ]:
best_model, best_params, best_score = tune_hyperparameters(
    X_train, y_train, X_val, y_val, seed
)

In [ ]:
feature_df = important_features(X_train, best_model)
print(feature_df)

In [ ]:
clf_report = evaluate_model(
    best_model, X_test, y_test,
    categorical_df, numeric_df,
    cat_mode, num_bounds, le_dict, scaler, le_y
)
print(clf_report)